In [1]:
# Cell 1 — Environment Check
import sys
import importlib

print(f"Python version: {sys.version}")

required = ["pandas", "numpy", "wbgapi", "requests"]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "version unknown")
        print(f"  {pkg}: {version}")
    except ImportError:
        print(f"  {pkg}: NOT FOUND")

Python version: 3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
  pandas: 2.3.3
  numpy: 1.26.4
  wbgapi: 1.0.14
  requests: 2.33.1


In [2]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import wbgapi as wb

print("Imports successful")

Imports successful


In [3]:
# Cell 3 — Nepal Fiscal Year Utility
# Nepal FY runs mid-July to mid-July
# Convention: FY2020 = July 2019 to July 2020
# World Bank calendar year X maps to Nepal FY X
# India fiscal year (Apr-Mar) ending March X maps to Nepal FY X

def to_nepal_fy(year, source="world_bank"):
    """
    Convert source year to Nepal fiscal year label.
    
    Parameters
    ----------
    year : int
        Calendar year or fiscal year from source
    source : str
        'world_bank' -> calendar year, maps directly to Nepal FY
        'india'      -> April-March FY, ending year maps to Nepal FY
    
    Returns
    -------
    int : Nepal FY year label
    """
    if source in ("world_bank", "india"):
        return int(year)
    else:
        raise ValueError(f"Unknown source: {source}")

# Quick sanity check
assert to_nepal_fy(2020, "world_bank") == 2020
assert to_nepal_fy(2020, "india") == 2020
print("Nepal FY utility ready")

Nepal FY utility ready


In [4]:
# Cell 4 — Fetch Nepal GDP Growth (F05)
# Source 1: World Bank indicator NY.GDP.MKTP.KD.ZG
# Coverage: 1995-2024
# Verification target: NRB reports

raw_gdp = wb.data.DataFrame(
    "NY.GDP.MKTP.KD.ZG",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

print(raw_gdp.shape)
print(raw_gdp.head(10))

(1, 30)
           YR1995    YR1996    YR1997    YR1998    YR1999  YR2000    YR2001  \
economy                                                                       
NPL      3.468452  5.328284  5.048613  3.016389  4.412573     6.2  4.799892   

           YR2002    YR2003    YR2004  ...    YR2015    YR2016    YR2017  \
economy                                ...                                 
NPL      0.120143  3.945038  4.682603  ...  3.976053  0.433114  8.977279   

           YR2018    YR2019    YR2020   YR2021    YR2022    YR2023    YR2024  
economy                                                                       
NPL      7.622376  6.657055 -2.369621  4.83815  5.631315  1.982548  3.665374  

[1 rows x 30 columns]


In [5]:
# Cell 5 — Reshape Nepal GDP Growth
gdp_growth = (
    raw_gdp
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "gdp_growth"})
)

gdp_growth["year"] = gdp_growth["year"].str.replace("YR", "").astype(int)
gdp_growth = gdp_growth.sort_values("year").reset_index(drop=True)

print(gdp_growth)

economy  year  gdp_growth
0        1995    3.468452
1        1996    5.328284
2        1997    5.048613
3        1998    3.016389
4        1999    4.412573
5        2000    6.200000
6        2001    4.799892
7        2002    0.120143
8        2003    3.945038
9        2004    4.682603
10       2005    3.479181
11       2006    3.364615
12       2007    3.411560
13       2008    6.104639
14       2009    4.533079
15       2010    4.816415
16       2011    3.421809
17       2012    4.670142
18       2013    3.525153
19       2014    6.011483
20       2015    3.976053
21       2016    0.433114
22       2017    8.977279
23       2018    7.622376
24       2019    6.657055
25       2020   -2.369621
26       2021    4.838150
27       2022    5.631315
28       2023    1.982548
29       2024    3.665374


In [6]:
# Cell 6 — Apply NRB Override for 2015
# FLAG: WB reported 3.98%, NRB reports 3.3%
# Discrepancy: 0.68% -> exceeds 0.5% threshold
# Resolution: NRB wins per project rule
# Source: NRB Annual Report 2015/16

gdp_growth.loc[gdp_growth["year"] == 2015, "gdp_growth"] = 3.3

# Confirm override
print(gdp_growth[gdp_growth["year"] == 2015])

economy  year  gdp_growth
20       2015         3.3


In [7]:
# Cell 7 — Fetch Nepal Inflation (F07)
# Source 1: World Bank indicator FP.CPI.TOTL.ZG
# Coverage: 1995-2024
# Verification target: NRB reports

raw_inflation = wb.data.DataFrame(
    "FP.CPI.TOTL.ZG",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

print(raw_inflation.shape)
print(raw_inflation)

(1, 30)
          YR1995    YR1996    YR1997     YR1998    YR1999   YR2000    YR2001  \
economy                                                                        
NPL      7.62297  9.220467  4.009989  11.244468  7.451113  2.47882  2.688304   

           YR2002    YR2003    YR2004  ...    YR2015    YR2016   YR2017  \
economy                                ...                                
NPL      3.029399  5.707009  2.841811  ...  7.868909  8.790343  2.77704   

           YR2018   YR2019    YR2020    YR2021    YR2022    YR2023    YR2024  
economy                                                                       
NPL      4.406594  5.56616  5.056358  4.126767  7.670206  7.115873  4.685097  

[1 rows x 30 columns]


In [8]:
# Cell 8 — Reshape Nepal Inflation and Apply NRB Overrides
inflation = (
    raw_inflation
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "inflation_yoy"})
)

inflation["year"] = inflation["year"].str.replace("YR", "").astype(int)
inflation = inflation.sort_values("year").reset_index(drop=True)

# FLAG: WB=8.79%, NRB=9.9%, discrepancy 1.11% -> NRB wins
# Source: NRB Annual Report 2016/17
inflation.loc[inflation["year"] == 2016, "inflation_yoy"] = 9.9

# FLAG: WB=7.67%, NRB=8.56%, discrepancy 0.89% -> NRB wins
# Source: NRB Annual Report 2022/23
inflation.loc[inflation["year"] == 2022, "inflation_yoy"] = 8.56

print(inflation)

economy  year  inflation_yoy
0        1995       7.622970
1        1996       9.220467
2        1997       4.009989
3        1998      11.244468
4        1999       7.451113
5        2000       2.478820
6        2001       2.688304
7        2002       3.029399
8        2003       5.707009
9        2004       2.841811
10       2005       6.836333
11       2006       6.920336
12       2007       2.269219
13       2008       9.907830
14       2009      11.094824
15       2010       9.326504
16       2011       9.227075
17       2012       9.459810
18       2013       9.040163
19       2014       8.364155
20       2015       7.868909
21       2016       9.900000
22       2017       2.777040
23       2018       4.406594
24       2019       5.566160
25       2020       5.056358
26       2021       4.126767
27       2022       8.560000
28       2023       7.115873
29       2024       4.685097


In [9]:
# Cell 9 — Fetch Forex Reserves Import Cover (F01)
# Source 1: World Bank indicator FI.RES.TOTL.MO
# Coverage: 1995-2024
# Verification target: NRB reports
# Unit: months of import cover

raw_forex = wb.data.DataFrame(
    "FI.RES.TOTL.MO",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

print(raw_forex.shape)
print(raw_forex)

(1, 30)
           YR1995    YR1996   YR1997    YR1998    YR1999    YR2000    YR2001  \
economy                                                                        
NPL      4.669444  4.257226  4.13729  6.569615  6.151447  6.491275  7.366591   

          YR2002    YR2003    YR2004  ...     YR2015     YR2016    YR2017  \
economy                               ...                                   
NPL      7.40749  7.713096  7.738908  ...  12.533761  10.288565  9.353803   

           YR2018    YR2019     YR2020    YR2021    YR2022     YR2023  \
economy                                                                 
NPL      6.614643  7.458683  12.479785  6.722983  7.118442  10.416933   

            YR2024  
economy             
NPL      12.971843  

[1 rows x 30 columns]


In [10]:
# Cell 10 — Reshape Forex and Apply NRB Overrides
forex = (
    raw_forex
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "forex_import_cover"})
)

forex["year"] = forex["year"].str.replace("YR", "").astype(int)
forex = forex.sort_values("year").reset_index(drop=True)

# FLAG: WB=12.48, NRB=11.8, discrepancy 0.68 -> NRB wins
# Source: NRB Annual Report 2020/21
forex.loc[forex["year"] == 2020, "forex_import_cover"] = 11.8

# FLAG: WB=12.97, NRB=12.0, discrepancy 0.97 -> NRB wins
# Source: NRB Annual Report 2024/25
forex.loc[forex["year"] == 2024, "forex_import_cover"] = 12.0

print(forex)

economy  year  forex_import_cover
0        1995            4.669444
1        1996            4.257226
2        1997            4.137290
3        1998            6.569615
4        1999            6.151447
5        2000            6.491275
6        2001            7.366591
7        2002            7.407490
8        2003            7.713096
9        2004            7.738908
10       2005            6.701434
11       2006            7.665498
12       2007            6.457352
13       2008            6.619583
14       2009            6.565363
15       2010            6.010522
16       2011            6.744538
17       2012            7.608926
18       2013            8.568363
19       2014            8.435011
20       2015           12.533761
21       2016           10.288565
22       2017            9.353803
23       2018            6.614643
24       2019            7.458683
25       2020           11.800000
26       2021            6.722983
27       2022            7.118442
28       2023 

In [11]:
# Cell 11 — Fetch Remittances (F02, F03)
# F02: remittance_growth_yoy — derived from levels
# F03: remittance_pct_gdp
# Source 1: World Bank indicator BX.TRF.PWKR.DT.GD.ZS (remittances % of GDP)
# Source 2: World Bank indicator BX.TRF.PWKR.CD.DT (remittances in USD, for growth calc)
# Verification target: NRB reports

raw_remit_pct = wb.data.DataFrame(
    "BX.TRF.PWKR.DT.GD.ZS",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

raw_remit_usd = wb.data.DataFrame(
    "BX.TRF.PWKR.CD.DT",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

print("Remittance % GDP:")
print(raw_remit_pct)
print("\nRemittance USD levels:")
print(raw_remit_usd)

Remittance % GDP:
         YR1995    YR1996    YR1997    YR1998    YR1999    YR2000    YR2001  \
economy                                                                       
NPL      1.2911  0.976652  1.005512  1.390061  1.658099  2.029361  2.446875   

           YR2002     YR2003    YR2004  ...     YR2015     YR2016     YR2017  \
economy                                 ...                                    
NPL      11.21302  12.180324  11.30899  ...  27.626085  26.960565  23.913544   

           YR2018    YR2019     YR2020     YR2021     YR2022     YR2023  \
economy                                                                   
NPL      25.02642  24.11637  24.250246  22.277579  22.853297  26.224109   

            YR2024  
economy             
NPL      26.225458  

[1 rows x 30 columns]

Remittance USD levels:
               YR1995        YR1996        YR1997        YR1998        YR1999  \
economy                                                                         
NPL  

In [12]:
# Cell 12 — Reshape Remittances
remit_pct = (
    raw_remit_pct
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "remittance_pct_gdp"})
)
remit_pct["year"] = remit_pct["year"].str.replace("YR", "").astype(int)
remit_pct = remit_pct.sort_values("year").reset_index(drop=True)

remit_usd = (
    raw_remit_usd
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "remittance_usd"})
)
remit_usd["year"] = remit_usd["year"].str.replace("YR", "").astype(int)
remit_usd = remit_usd.sort_values("year").reset_index(drop=True)

# Compute YoY growth from USD levels
remit_usd["remittance_growth_yoy"] = remit_usd["remittance_usd"].pct_change() * 100

# Merge
remittances = remit_pct.merge(
    remit_usd[["year", "remittance_growth_yoy"]],
    on="year"
)

print(remittances)

economy  year  remittance_pct_gdp  remittance_growth_yoy
0        1995            1.291100                    NaN
1        1996            0.976652             -22.284313
2        1997            1.005512              11.997094
3        1998            1.390061              36.489191
4        1999            1.658099              23.639583
5        2000            2.029361              33.590316
6        2001            2.446875              31.827386
7        2002           11.213020             361.601654
8        2003           12.180324              13.646007
9        2004           11.308990               6.683788
10       2005           14.905101              47.314620
11       2006           16.068965              19.921071
12       2007           16.791813              19.310558
13       2008           21.738104              57.287371
14       2009           23.207625               9.394323
15       2010           21.646985              16.114701
16       2011           19.5463

In [13]:
# Cell 13 — Fix 2002 Remittance Growth Anomaly
# FLAG: 2002 shows 361% YoY growth
# Reason: World Bank series has structural break pre-2003
# NRB remittance tracking became systematic only after 2002
# Resolution: set 2002 growth to NaN, do not use for modeling
# This affects F02 only, F03 (pct_gdp) is unaffected

remittances.loc[remittances["year"] == 2002, "remittance_growth_yoy"] = np.nan

print(remittances[["year", "remittance_growth_yoy"]].head(15))

economy  year  remittance_growth_yoy
0        1995                    NaN
1        1996             -22.284313
2        1997              11.997094
3        1998              36.489191
4        1999              23.639583
5        2000              33.590316
6        2001              31.827386
7        2002                    NaN
8        2003              13.646007
9        2004               6.683788
10       2005              47.314620
11       2006              19.921071
12       2007              19.310558
13       2008              57.287371
14       2009               9.394323


In [14]:
# Cell 14 — Fetch India GDP Growth (F09, F10)
# Source 1: World Bank indicator NY.GDP.MKTP.KD.ZG
# Coverage: 1995-2024
# India FY: April-March, ending year maps to Nepal FY per fiscal year rule
# F09: india_gdp_growth_lag1 (derived later)
# F10: india_gdp_growth_lag2 (derived later)

raw_india_gdp = wb.data.DataFrame(
    "NY.GDP.MKTP.KD.ZG",
    economy="IND",
    time=range(1995, 2025),
    labels=False
)

print(raw_india_gdp.shape)
print(raw_india_gdp)

(1, 30)
           YR1995    YR1996    YR1997    YR1998    YR1999    YR2000    YR2001  \
economy                                                                         
IND      7.574492  7.549522  4.049821  6.184416  8.845756  3.840991  4.823966   

           YR2002    YR2003    YR2004  ...    YR2015    YR2016    YR2017  \
economy                                ...                                 
IND      3.803975  7.860381  7.922937  ...  7.996254  8.256306  6.795383   

           YR2018    YR2019    YR2020    YR2021    YR2022    YR2023    YR2024  
economy                                                                        
IND      6.453851  3.871437 -5.777725  9.689592  7.609365  9.190755  6.494766  

[1 rows x 30 columns]


In [15]:
# Cell 15 — Reshape India GDP Growth and Apply IMF Overrides
india_gdp = (
    raw_india_gdp
    .T
    .reset_index()
    .rename(columns={"index": "year", "IND": "india_gdp_growth"})
)

india_gdp["year"] = india_gdp["year"].str.replace("YR", "").astype(int)
india_gdp = india_gdp.sort_values("year").reset_index(drop=True)

# FLAG: WB=-5.78%, IMF=-6.6%, discrepancy 0.82% -> IMF wins
# Source: IMF World Economic Outlook 2021
india_gdp.loc[india_gdp["year"] == 2020, "india_gdp_growth"] = -6.6

# FLAG: WB=9.69%, IMF=8.9%, discrepancy 0.79% -> IMF wins
# Source: IMF World Economic Outlook 2022
india_gdp.loc[india_gdp["year"] == 2021, "india_gdp_growth"] = 8.9

print(india_gdp)

economy  year  india_gdp_growth
0        1995          7.574492
1        1996          7.549522
2        1997          4.049821
3        1998          6.184416
4        1999          8.845756
5        2000          3.840991
6        2001          4.823966
7        2002          3.803975
8        2003          7.860381
9        2004          7.922937
10       2005          7.923431
11       2006          8.060733
12       2007          7.660815
13       2008          3.086698
14       2009          7.861889
15       2010          8.497585
16       2011          5.241316
17       2012          5.456388
18       2013          6.386106
19       2014          7.410228
20       2015          7.996254
21       2016          8.256306
22       2017          6.795383
23       2018          6.453851
24       2019          3.871437
25       2020         -6.600000
26       2021          8.900000
27       2022          7.609365
28       2023          9.190755
29       2024          6.494766


In [16]:
# Cell 16 — Fetch India Inflation (F13)
# Source 1: World Bank indicator FP.CPI.TOTL.ZG
# Coverage: 1995-2024
# Used for nepal_india_inflation_spread (F14)
# Verification target: IMF World Economic Outlook

raw_india_inflation = wb.data.DataFrame(
    "FP.CPI.TOTL.ZG",
    economy="IND",
    time=range(1995, 2025),
    labels=False
)

print(raw_india_inflation.shape)
print(raw_india_inflation)

(1, 30)
            YR1995    YR1996    YR1997     YR1998   YR1999    YR2000  \
economy                                                                
IND      10.224886  8.977152  7.164252  13.230839  4.66982  4.009436   

           YR2001    YR2002    YR2003    YR2004  ...    YR2015    YR2016  \
economy                                          ...                       
IND      3.779293  4.297152  3.805859  3.767252  ...  4.906973  4.948216   

           YR2017    YR2018    YR2019    YR2020    YR2021    YR2022    YR2023  \
economy                                                                         
IND      3.328173  3.938826  3.729506  6.623437  5.131407  6.699034  5.649143   

           YR2024  
economy            
IND      4.953036  

[1 rows x 30 columns]


In [17]:
# Cell 17 — Reshape India Inflation
india_inflation = (
    raw_india_inflation
    .T
    .reset_index()
    .rename(columns={"index": "year", "IND": "india_inflation"})
)

india_inflation["year"] = india_inflation["year"].str.replace("YR", "").astype(int)
india_inflation = india_inflation.sort_values("year").reset_index(drop=True)

print(india_inflation)

economy  year  india_inflation
0        1995        10.224886
1        1996         8.977152
2        1997         7.164252
3        1998        13.230839
4        1999         4.669820
5        2000         4.009436
6        2001         3.779293
7        2002         4.297152
8        2003         3.805859
9        2004         3.767252
10       2005         4.246344
11       2006         5.796523
12       2007         6.372881
13       2008         8.349267
14       2009        10.882353
15       2010        11.989390
16       2011         8.911793
17       2012         9.478997
18       2013        10.017878
19       2014         6.665657
20       2015         4.906973
21       2016         4.948216
22       2017         3.328173
23       2018         3.938826
24       2019         3.729506
25       2020         6.623437
26       2021         5.131407
27       2022         6.699034
28       2023         5.649143
29       2024         4.953036


In [18]:
# Cell 18 — Fetch Current Account % GDP (F08)
# Source 1: World Bank indicator BN.CAB.XOKA.GD.ZS
# Coverage: 1995-2024
# Verification target: IMF World Economic Outlook
# Note: negative value = deficit

raw_current_account = wb.data.DataFrame(
    "BN.CAB.XOKA.GD.ZS",
    economy="NPL",
    time=range(1995, 2025),
    labels=False
)

print(raw_current_account.shape)
print(raw_current_account)

(1, 30)
           YR1995   YR1996    YR1997    YR1998    YR1999    YR2000    YR2001  \
economy                                                                        
NPL     -8.097741 -7.22315 -7.890015 -1.383662 -1.563039 -2.377937 -2.741571   

           YR2002    YR2003    YR2004  ...     YR2015    YR2016    YR2017  \
economy                                ...                                  
NPL      3.556297  2.847594  1.374222  ...  10.043308 -0.684347 -3.564329   

           YR2018    YR2019    YR2020     YR2021   YR2022    YR2023    YR2024  
economy                                                                        
NPL     -8.300805 -5.129631 -0.251654 -14.524016 -5.76693  2.450495  3.912792  

[1 rows x 30 columns]


In [19]:
# Cell 19 — Reshape Current Account and Apply IMF Override
current_account = (
    raw_current_account
    .T
    .reset_index()
    .rename(columns={"index": "year", "NPL": "current_account_pct_gdp"})
)

current_account["year"] = current_account["year"].str.replace("YR", "").astype(int)
current_account = current_account.sort_values("year").reset_index(drop=True)

# FLAG: WB=-14.52%, IMF=-12.8%, discrepancy 1.72% -> IMF wins
# Source: IMF World Economic Outlook 2022
current_account.loc[current_account["year"] == 2021, "current_account_pct_gdp"] = -12.8

print(current_account)

economy  year  current_account_pct_gdp
0        1995                -8.097741
1        1996                -7.223150
2        1997                -7.890015
3        1998                -1.383662
4        1999                -1.563039
5        2000                -2.377937
6        2001                -2.741571
7        2002                 3.556297
8        2003                 2.847594
9        2004                 1.374222
10       2005                 1.883024
11       2006                 1.659502
12       2007                 0.054810
13       2008                 5.845486
14       2009                 0.166582
15       2010                -0.797466
16       2011                 1.337751
17       2012                 2.660707
18       2013                 5.231374
19       2014                 2.182982
20       2015                10.043308
21       2016                -0.684347
22       2017                -3.564329
23       2018                -8.300805
24       2019            

In [20]:
# Cell 20 — Merge All Indicators into Master DataFrame
master = gdp_growth.copy()

master = master.merge(inflation, on="year")
master = master.merge(forex, on="year")
master = master.merge(remittances, on="year")
master = master.merge(india_gdp, on="year")
master = master.merge(india_inflation, on="year")
master = master.merge(current_account, on="year")

# Drop economy column if present
if "economy" in master.columns:
    master = master.drop(columns=["economy"])

print(master.shape)
print(master.columns.tolist())
print(master)

(30, 9)
['year', 'gdp_growth', 'inflation_yoy', 'forex_import_cover', 'remittance_pct_gdp', 'remittance_growth_yoy', 'india_gdp_growth', 'india_inflation', 'current_account_pct_gdp']
economy  year  gdp_growth  inflation_yoy  forex_import_cover  \
0        1995    3.468452       7.622970            4.669444   
1        1996    5.328284       9.220467            4.257226   
2        1997    5.048613       4.009989            4.137290   
3        1998    3.016389      11.244468            6.569615   
4        1999    4.412573       7.451113            6.151447   
5        2000    6.200000       2.478820            6.491275   
6        2001    4.799892       2.688304            7.366591   
7        2002    0.120143       3.029399            7.407490   
8        2003    3.945038       5.707009            7.713096   
9        2004    4.682603       2.841811            7.738908   
10       2005    3.479181       6.836333            6.701434   
11       2006    3.364615       6.920336         

In [21]:
# Cell 21 — Derive Crisis Labels
# LOCKED thresholds:
# T1: GDP growth < 2%
# T2: Forex < 7 months import cover
# T3: Inflation > 8%
# T4: Remittance drop > 20% YoY
# T5: Current account deficit > 10% of GDP
# Rule: any 2 of 5 = crisis year (label = 1)

master["t1_gdp"] = (master["gdp_growth"] < 2).astype(int)
master["t2_forex"] = (master["forex_import_cover"] < 7).astype(int)
master["t3_inflation"] = (master["inflation_yoy"] > 8).astype(int)
master["t4_remittance"] = (master["remittance_growth_yoy"] < -20).astype(int)
master["t5_current_account"] = (master["current_account_pct_gdp"] < -10).astype(int)

master["triggers_fired"] = (
    master["t1_gdp"] +
    master["t2_forex"] +
    master["t3_inflation"] +
    master["t4_remittance"] +
    master["t5_current_account"]
)

master["crisis"] = (master["triggers_fired"] >= 2).astype(int)

# Review
cols = ["year", "t1_gdp", "t2_forex", "t3_inflation", "t4_remittance", 
        "t5_current_account", "triggers_fired", "crisis"]
print(master[cols].to_string())

economy  year  t1_gdp  t2_forex  t3_inflation  t4_remittance  t5_current_account  triggers_fired  crisis
0        1995       0         1             0              0                   0               1       0
1        1996       0         1             1              1                   0               3       1
2        1997       0         1             0              0                   0               1       0
3        1998       0         1             1              0                   0               2       1
4        1999       0         1             0              0                   0               1       0
5        2000       0         1             0              0                   0               1       0
6        2001       0         0             0              0                   0               0       0
7        2002       1         0             0              0                   0               1       0
8        2003       0         0             0          

In [22]:
# Cell 22 — Apply NRB Inflation Override for 2023
# FLAG: WB=7.12%, NRB=7.74%, discrepancy 0.62% -> NRB wins
# Source: NRB Annual Report 2022/23
# Note: still below 8% threshold, so crisis label for 2023 unchanged

inflation.loc[inflation["year"] == 2023, "inflation_yoy"] = 7.74
master.loc[master["year"] == 2023, "inflation_yoy"] = 7.74

# Confirm
print(master[master["year"] == 2023][["year", "inflation_yoy", "t3_inflation", "crisis"]])


economy  year  inflation_yoy  t3_inflation  crisis
28       2023           7.74             0       0


In [23]:
# Cell 23 — Final Crisis Label Review
crisis_years = master[master["crisis"] == 1]["year"].tolist()
non_crisis = master[master["crisis"] == 0]["year"].tolist()

print(f"Total years: {len(master)}")
print(f"Crisis years ({len(crisis_years)}): {crisis_years}")
print(f"Non-crisis years ({len(non_crisis)}): {non_crisis}")
print(f"Crisis rate: {len(crisis_years)/len(master)*100:.1f}%")
print()
print(master[["year","gdp_growth","inflation_yoy","forex_import_cover",
              "t1_gdp","t2_forex","t3_inflation","t4_remittance",
              "t5_current_account","triggers_fired","crisis"]].to_string())

Total years: 30
Crisis years (8): [1996, 1998, 2008, 2009, 2010, 2011, 2016, 2021]
Non-crisis years (22): [1995, 1997, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020, 2022, 2023, 2024]
Crisis rate: 26.7%

economy  year  gdp_growth  inflation_yoy  forex_import_cover  t1_gdp  t2_forex  t3_inflation  t4_remittance  t5_current_account  triggers_fired  crisis
0        1995    3.468452       7.622970            4.669444       0         1             0              0                   0               1       0
1        1996    5.328284       9.220467            4.257226       0         1             1              1                   0               3       1
2        1997    5.048613       4.009989            4.137290       0         1             0              0                   0               1       0
3        1998    3.016389      11.244468            6.569615       0         1             1              0                   0      

In [27]:
# Cell 24 — Save Master DataFrame
# verified_data.csv is sacred — never overwrite
# This is the raw + labeled dataset before feature engineering

master.to_csv("../data/master_labeled.csv", index=False)
print("Saved: ../data/master_labeled.csv")
print(f"Shape: {master.shape}")
print(f"Crisis years: {master[master['crisis']==1]['year'].tolist()}")

Saved: ../data/master_labeled.csv
Shape: (30, 16)
Crisis years: [1996, 1998, 2008, 2009, 2010, 2011, 2016, 2021]
